# 🏠 Dự đoán Giá Nhà — Buổi 7/9
## Model Evaluation: Đánh Giá Model Toàn Diện

---

### 🗺️ Lộ trình 9 buổi

| # | Chủ đề | Trạng thái |
|---|--------|-----------|
| 1 | Giới thiệu & Khám Phá Dữ Liệu (EDA) | ✅ Hoàn thành |
| 2 | Phân Tích Thống Kê & Trực Quan Hóa | ✅ Hoàn thành |
| 3 | Data Cleaning & Preprocessing | ✅ Hoàn thành |
| 4 | Feature Engineering | ✅ Hoàn thành |
| 5 | Feature Selection & Dimensionality Reduction | ✅ Hoàn thành |
| 6 | Model Training (Linear + Tree + Hyperparameters + Stacking) | ✅ Hoàn thành |
| **7** | **Model Evaluation — Đánh Giá Toàn Diện** | 🔄 **Buổi này** |
| 8 | Kaggle Submission & Cải Thiện Kết Quả | ⏳ Tiếp theo |
| 9 | Tổng Kết & Hướng Đi Tiếp Theo | ⏳ |

---

### 🎯 Mục tiêu buổi 7

Sau buổi này bạn sẽ biết:
1. **Metrics:** RMSE, MAE, R² — đọc và hiểu từng chỉ số
2. **Cross-Validation:** Phân tích RMSE từng fold để hiểu độ ổn định
3. **Learning Curves:** Phát hiện model đang bị overfit hay underfit
4. **Residual Analysis:** Xem model còn sai ở đâu
5. **So sánh tổng quát:** Biểu đồ + bảng đánh giá tất cả models

In [ ]:
# ============================================================
# ♻️ SETUP — Import thư viện & rebuild pipeline từ Buổi 5
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# --- Linear models ---
from sklearn.linear_model import RidgeCV, LassoCV, Ridge

# --- Tree-based models ---
from sklearn.ensemble import RandomForestRegressor, StackingRegressor

# --- XGBoost (cần cài: pip install xgboost) ---
try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
    print("✅ XGBoost sẵn sàng")
except ImportError:
    HAVE_XGB = False
    print("⚠️  XGBoost chưa cài — cài bằng: pip install xgboost")

# --- Đánh giá ---
from sklearn.model_selection import cross_val_score, KFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# ── Load dữ liệu ──────────────────────────────────────────
train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')

train_id = train_df['Id'].copy()
test_id  = test_df['Id'].copy()
y = np.log1p(train_df['SalePrice'])

train_df.drop(['Id', 'SalePrice'], axis=1, inplace=True)
test_df.drop(['Id'], axis=1, inplace=True)

ntrain = len(train_df)
ntest  = len(test_df)
all_data = pd.concat([train_df, test_df], axis=0, ignore_index=True)

# ── Fill missing ──────────────────────────────────────────
none_cols = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu',
             'GarageType','GarageFinish','GarageQual','GarageCond',
             'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','MasVnrType']
zero_cols = ['GarageYrBlt','GarageArea','GarageCars','MasVnrArea',
             'BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath']
mode_cols = ['MSZoning','Utilities','Functional','Electrical',
             'KitchenQual','Exterior1st','Exterior2nd','SaleType']

for col in none_cols: all_data[col] = all_data[col].fillna('None')
for col in zero_cols: all_data[col] = all_data[col].fillna(0)
all_data['LotFrontage'] = (all_data.groupby('Neighborhood')['LotFrontage']
                           .transform(lambda x: x.fillna(x.median())))
for col in mode_cols: all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# ── Xoá outlier ──────────────────────────────────────────
outlier_mask = (all_data['GrLivArea'][:ntrain] > 4000) & (np.expm1(y) < 300_000)
outlier_idx  = outlier_mask[outlier_mask].index.tolist()
all_data.drop(index=outlier_idx, inplace=True)
all_data.reset_index(drop=True, inplace=True)
y.drop(index=outlier_idx, inplace=True)
y.reset_index(drop=True, inplace=True)
ntrain = ntrain - len(outlier_idx)

# ── Feature Engineering ───────────────────────────────────
all_data['TotalSF']      = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalPorchSF'] = (all_data['OpenPorchSF'] + all_data['3SsnPorch'] +
                             all_data['EnclosedPorch'] + all_data['ScreenPorch'] + all_data['WoodDeckSF'])
all_data['TotalBath']    = (all_data['FullBath'] + 0.5*all_data['HalfBath'] +
                             all_data['BsmtFullBath'] + 0.5*all_data['BsmtHalfBath'])
all_data['OverallScore'] = all_data['OverallQual'] * all_data['OverallCond']
all_data['HasGarage']    = (all_data['GarageArea'] > 0).astype(int)
all_data['HasBsmt']      = (all_data['TotalBsmtSF'] > 0).astype(int)
all_data['HouseAge']     = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge']     = all_data['YrSold'] - all_data['YearRemodAdd']

for feat in ['OverallQual', 'GrLivArea', 'TotalSF']:
    all_data[f'{feat}_sq']   = all_data[feat] ** 2
    all_data[f'{feat}_sqrt'] = np.sqrt(all_data[feat])

for feat in ['GrLivArea', 'TotalSF', 'LotArea']:
    all_data[f'{feat}_log'] = np.log1p(all_data[feat])

# ── Ordinal Encoding ──────────────────────────────────────
ordinal_mappings = {
    'ExterQual':    ['None','Po','Fa','TA','Gd','Ex'],
    'ExterCond':    ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtQual':     ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtCond':     ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtExposure': ['None','No','Mn','Av','Gd'],
    'BsmtFinType1': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'BsmtFinType2': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'HeatingQC':    ['None','Po','Fa','TA','Gd','Ex'],
    'KitchenQual':  ['None','Po','Fa','TA','Gd','Ex'],
    'FireplaceQu':  ['None','Po','Fa','TA','Gd','Ex'],
    'GarageFinish': ['None','Unf','RFn','Fin'],
    'GarageQual':   ['None','Po','Fa','TA','Gd','Ex'],
    'GarageCond':   ['None','Po','Fa','TA','Gd','Ex'],
    'PoolQC':       ['None','Fa','TA','Gd','Ex'],
    'Fence':        ['None','MnWw','GdWo','MnPrv','GdPrv'],
    'Functional':   ['Sal','Sev','Maj2','Maj1','Mod','Min2','Min1','Typ'],
    'LandSlope':    ['Sev','Mod','Gtl'],
    'LotShape':     ['IR3','IR2','IR1','Reg'],
    'PavedDrive':   ['N','P','Y'],
}
for col, order in ordinal_mappings.items():
    mapping = {val: i for i, val in enumerate(order)}
    all_data[col] = all_data[col].map(mapping).fillna(0).astype(int)

# ── One-Hot + Variance Filter + Scale ────────────────────
cat_cols     = all_data.select_dtypes(include=['object']).columns.tolist()
all_data_enc = pd.get_dummies(all_data, columns=cat_cols, drop_first=True)

X_train_raw = all_data_enc[:ntrain].copy()
X_test_raw  = all_data_enc[ntrain:].copy()
X_test_raw.reset_index(drop=True, inplace=True)

selector      = VarianceThreshold(threshold=0.01)
selected_mask = selector.fit(X_train_raw).get_support()
X_train_sel   = X_train_raw.loc[:, selected_mask]
X_test_sel    = X_test_raw.loc[:, selected_mask]

scaler  = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train_sel),
                        columns=X_train_sel.columns)
X_test  = pd.DataFrame(scaler.transform(X_test_sel),
                        columns=X_test_sel.columns)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def rmse_cv(model):
    scores = cross_val_score(model, X_train, y,
                             scoring='neg_root_mean_squared_error', cv=kf)
    return -scores.mean()

# ── Train các model để dùng trong buổi này ───────────────
print("Training models (chờ vài giây)...")

ridge = RidgeCV(alphas=[1, 10, 50, 100, 300])
ridge.fit(X_train, y)
ridge_rmse = rmse_cv(ridge)

lasso = LassoCV(alphas=[0.0001, 0.001, 0.01], cv=kf, max_iter=5000)
lasso.fit(X_train, y)
lasso_rmse = rmse_cv(lasso)

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y)
rf_rmse = rmse_cv(rf)

if HAVE_XGB:
    xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                        subsample=0.8, random_state=42, verbosity=0)
    xgb.fit(X_train, y)
    xgb_rmse = rmse_cv(xgb)

# Model tốt nhất để dùng trong phân tích
best_model      = xgb if HAVE_XGB else rf
best_model_name = "XGBoost" if HAVE_XGB else "Random Forest"

print("✅ Sẵn sàng!")
print(f"   Ridge         RMSE = {ridge_rmse:.4f}")
print(f"   Lasso         RMSE = {lasso_rmse:.4f}")
print(f"   Random Forest RMSE = {rf_rmse:.4f}")
if HAVE_XGB:
    print(f"   XGBoost       RMSE = {xgb_rmse:.4f}")
print(f"\n   → Model tốt nhất dùng để phân tích: {best_model_name}")

> ⬆️ **Chạy cell Setup trước** — toàn bộ pipeline và models sẽ được chuẩn bị tự động.

---

## 📏 Task 1 — Các Chỉ Số Đánh Giá: RMSE, MAE, R²

Khi model đưa ra dự đoán, ta cần **đo lường sai số** để biết model tốt đến đâu.

### Ba chỉ số phổ biến nhất

#### RMSE — Root Mean Squared Error
$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

- Phạt nặng hơn với **lỗi lớn** (vì bình phương)
- Dùng cho bài toán khi lỗi lớn nguy hiểm hơn lỗi nhỏ

#### MAE — Mean Absolute Error
$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

- Dễ hiểu hơn: "trung bình mỗi dự đoán sai bao nhiêu"
- Không phạt nặng outliers

#### R² — Coefficient of Determination
$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

- R² = 1.0 → dự đoán hoàn hảo
- R² = 0.0 → model không tốt hơn chỉ đơn giản là đoán trung bình
- Thường thấy R² ~ 0.87–0.92 là tốt cho bài này

> **Lưu ý:** Vì y đã được log-transform, RMSE/MAE tính trên log scale. RMSE=0.12 ≈ sai ~12% theo thang log.

In [ ]:
# ── Task 1: Tính metrics cho từng model ───────────────────
def get_metrics(model, name):
    """Tính RMSE, MAE, R² trên training data."""
    y_pred = model.predict(X_train)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae  = mean_absolute_error(y, y_pred)
    r2   = r2_score(y, y_pred)
    return {'Model': name, 'RMSE (train)': rmse, 'MAE (train)': mae, 'R² (train)': r2}

rows = [
    get_metrics(ridge, 'Ridge'),
    get_metrics(lasso, 'Lasso'),
    get_metrics(rf,    'Random Forest'),
]
if HAVE_XGB:
    rows.append(get_metrics(xgb, 'XGBoost'))

metrics_df = pd.DataFrame(rows).set_index('Model')
print("📊 Metrics trên Training Data (thấp RMSE, cao R² = tốt hơn):")
print(metrics_df.to_string())

print("\n⚠️  Train RMSE thường thấp hơn CV RMSE — vì model đã 'học' training data!")
print("    → Dùng CV RMSE để đánh giá khả năng tổng quát hóa thực sự.")
print(f"\n📊 CV RMSE (thực tế hơn):")
print(f"   Ridge         = {ridge_rmse:.4f}")
print(f"   Lasso         = {lasso_rmse:.4f}")
print(f"   Random Forest = {rf_rmse:.4f}")
if HAVE_XGB:
    print(f"   XGBoost       = {xgb_rmse:.4f}")

### 💡 Khi nào dùng chỉ số nào?

| Tình huống | Nên dùng |
|-----------|---------|
| So sánh models với nhau | **RMSE** hoặc **CV RMSE** |
| Giải thích cho người không kỹ thuật | **MAE** (dễ hiểu hơn) |
| Muốn biết model giải thích được bao nhiêu % variance | **R²** |
| Submit Kaggle (bài này) | **RMSE trên log SalePrice** |

**Quy tắc vàng:** Luôn dùng **CV RMSE** (không phải train RMSE) khi so sánh models!

In [ ]:
---

## 🔁 Task 2 — Cross-Validation: Phân Tích Từng Fold

### Tại sao cần xem từng fold?

Nếu chỉ xem **trung bình CV RMSE**, ta có thể bỏ sót vấn đề:

```
Fold 1: RMSE = 0.110  ← tốt
Fold 2: RMSE = 0.112  ← tốt
Fold 3: RMSE = 0.180  ← đột nhiên xấu! Có gì đó sai?
Fold 4: RMSE = 0.111  ← tốt
Fold 5: RMSE = 0.108  ← tốt
Trung bình = 0.124 ← che giấu vấn đề!
```

Xem từng fold giúp ta biết model có **ổn định** không hay bị ảnh hưởng bởi 1 phần data.

# ── Task 2: Phân tích CV từng fold ────────────────────────
scores = cross_val_score(best_model, X_train, y,
                         scoring='neg_root_mean_squared_error', cv=kf)
cv_rmses = -scores

print(f"CV RMSE từng fold — {best_model_name}:")
for i, s in enumerate(cv_rmses, 1):
    bar = '█' * int(s * 500)
    print(f"  Fold {i}: {s:.4f}  {bar}")
print(f"\n  Trung bình : {cv_rmses.mean():.4f}")
print(f"  Độ lệch    : {cv_rmses.std():.4f}  (nhỏ = model ổn định)")

# Biểu đồ bar từng fold
plt.figure(figsize=(7, 4))
colors = ['#e74c3c' if s > cv_rmses.mean() + cv_rmses.std() else '#3498db'
          for s in cv_rmses]
plt.bar([f'Fold {i}' for i in range(1, 6)], cv_rmses, color=colors)
plt.axhline(cv_rmses.mean(), color='red', linestyle='--', linewidth=1.5,
            label=f'Trung bình = {cv_rmses.mean():.4f}')
plt.title(f'CV RMSE từng Fold — {best_model_name}', fontsize=12)
plt.ylabel('RMSE')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
### 💡 Đọc kết quả

| Tình huống | Ý nghĩa |
|-----------|---------|
| Độ lệch thấp (< 0.005) | Model ổn định — tin tưởng được |
| Độ lệch cao (> 0.015) | Model nhạy cảm với data — cần kiểm tra |
| 1 fold xấu hơn hẳn | Có thể có outliers hoặc pattern đặc biệt trong fold đó |

---

## 📈 Task 3 — Learning Curves: Overfit hay Underfit?

### Bias vs. Variance — hai vấn đề phổ biến nhất

```
UNDERFIT (High Bias)          OVERFIT (High Variance)
Model quá đơn giản            Model quá phức tạp

Train RMSE: cao               Train RMSE: rất thấp
Val   RMSE: cao               Val   RMSE: cao hơn nhiều
```

### Learning Curve là gì?

Ta train model với **số lượng data tăng dần** và xem Train RMSE và Validation RMSE thay đổi thế nào:

- **Underfit:** Cả hai đều cao và gần nhau
- **Overfit:** Train thấp, Validation cao — khoảng cách lớn
- **Tốt:** Cả hai thấp và gần nhau

In [ ]:
# ── Task 3: Learning Curves ────────────────────────────────
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y,
    train_sizes = np.linspace(0.2, 1.0, 6),   # 20%, 36%, ..., 100% của data
    scoring     = 'neg_root_mean_squared_error',
    cv          = 5,
    n_jobs      = -1
)

train_rmse = -train_scores.mean(axis=1)
val_rmse   = -val_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_rmse, 'o-', color='#3498db', label='Train RMSE')
plt.plot(train_sizes, val_rmse,   'o-', color='#e74c3c', label='Validation RMSE')
plt.fill_between(train_sizes, train_rmse, val_rmse, alpha=0.1, color='gray')
plt.xlabel('Số lượng training samples')
plt.ylabel('RMSE')
plt.title(f'Learning Curves — {best_model_name}', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

gap = val_rmse[-1] - train_rmse[-1]
print(f"Khoảng cách Train vs Val RMSE (khi dùng toàn bộ data): {gap:.4f}")
if gap > 0.02:
    print("→ Model đang bị OVERFIT — thử giảm độ phức tạp hoặc thêm regularization")
elif gap < 0.005:
    print("→ Model có thể UNDERFIT — thử tăng độ phức tạp")
else:
    print("→ Model khá cân bằng ✅")

### 💡 Đọc Learning Curves

| Hình dạng | Chẩn đoán | Giải pháp |
|-----------|-----------|-----------|
| Cả hai đường cao & gần nhau | Underfit | Thêm features, dùng model phức tạp hơn |
| Train thấp, Val cao (khoảng cách lớn) | Overfit | Thêm data, giảm complexity, regularize |
| Cả hai giảm khi thêm data | Đang học tốt | Thêm nhiều data sẽ tiếp tục cải thiện |
| Val plateau (không giảm nữa) | Đã tối ưu | Cần thay đổi features hoặc model |

In [ ]:
---

## 🔎 Task 4 — Residual Analysis: Model Còn Sai Ở Đâu?

**Residual** = Giá trị thực tế − Dự đoán = $y_i - \hat{y}_i$

### Model tốt thì residuals phải:
1. **Phân phối ngẫu nhiên** quanh 0 — không có pattern
2. **Phân phối chuẩn** (hình chuông) — không lệch một bên
3. **Độc lập với giá trị dự đoán** — sai số không tăng theo predicted

Nếu thấy pattern trong residuals → model chưa nắm bắt được một mối quan hệ nào đó!

# ── Task 4: Residual plots ─────────────────────────────────
y_pred    = best_model.predict(X_train)
residuals = y - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Residuals vs Predicted
axes[0].scatter(y_pred, residuals, alpha=0.3, s=15, color='#3498db')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Predicted (log SalePrice)')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted')

# Plot 2: Histogram của residuals
axes[1].hist(residuals, bins=40, color='#2ecc71', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Số lượng')
axes[1].set_title('Phân phối Residuals')

plt.tight_layout()
plt.show()

print(f"Residuals — {best_model_name}:")
print(f"  Trung bình : {residuals.mean():.4f}  (gần 0 = tốt)")
print(f"  Độ lệch    : {residuals.std():.4f}")
print(f"  Min/Max    : {residuals.min():.4f} / {residuals.max():.4f}")

In [ ]:
### 💡 Đọc Residual Plots

**Plot 1 — Residuals vs Predicted:**
- ✅ Tốt: Các điểm rải đều quanh đường y=0, không có pattern
- ❌ Xấu: Hình phễu (variance tăng), hình cung (bỏ sót non-linear)

**Plot 2 — Histogram:**
- ✅ Tốt: Hình chuông đối xứng quanh 0
- ❌ Xấu: Lệch trái/phải → model đang đoán thấp hoặc cao hơn thực tế

---

## 🏆 Task 5 — So Sánh Tất Cả Models

Cuối buổi, ta tập hợp kết quả của tất cả models vào 1 bảng và biểu đồ để có cái nhìn tổng quan.

In [ ]:
# ── Task 5: Final Model Comparison ────────────────────────
model_names = ['Ridge', 'Lasso', 'Random Forest']
cv_rmses    = [ridge_rmse, lasso_rmse, rf_rmse]

if HAVE_XGB:
    model_names.append('XGBoost')
    cv_rmses.append(xgb_rmse)

# Bảng so sánh
comparison = pd.DataFrame({'Model': model_names, 'CV RMSE': cv_rmses})
comparison = comparison.sort_values('CV RMSE').reset_index(drop=True)
comparison.index += 1
print("📊 Bảng xếp hạng Models (CV RMSE — thấp hơn = tốt hơn):")
print(comparison.to_string())

# Biểu đồ bar nằm ngang
colors = ['#27ae60' if i == 0 else '#3498db' for i in range(len(model_names))]
sorted_names  = comparison['Model'].tolist()
sorted_rmses  = comparison['CV RMSE'].tolist()
bar_colors    = ['#27ae60'] + ['#3498db'] * (len(sorted_names) - 1)

plt.figure(figsize=(8, 4))
bars = plt.barh(sorted_names, sorted_rmses, color=bar_colors)
for bar, val in zip(bars, sorted_rmses):
    plt.text(val + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontweight='bold')
plt.xlabel('CV RMSE')
plt.title('So Sánh Models — CV RMSE (thấp hơn = tốt hơn)', fontsize=12)
plt.xlim(0, max(sorted_rmses) * 1.1)
plt.tight_layout()
plt.show()

best = comparison.iloc[0]
print(f"\n🥇 Model tốt nhất: {best['Model']} với CV RMSE = {best['CV RMSE']:.4f}")

---

## 📝 Bài Tập

### Bài 1 — Tính CV RMSE cho Stacking

Ở buổi 6 bạn đã train `stacking` model. Hãy:
1. Train lại stacking model ở đây
2. Tính CV RMSE và thêm vào bảng so sánh

### Bài 2 — So sánh Ridge vs Lasso qua Residuals

Vẽ residual plot của cả Ridge và Lasso cạnh nhau, nhận xét sự khác biệt.

> **Gợi ý:** Dùng `fig, axes = plt.subplots(1, 2, ...)` và vẽ mỗi model vào một subplot

In [ ]:
# ── Bài Tập 1: Stacking CV RMSE ───────────────────────────
if HAVE_XGB:
    base_models = [
        ('ridge', Ridge(alpha=50)),
        ('rf',    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
        ('xgb',   XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                random_state=42, verbosity=0)),
    ]
else:
    base_models = [
        ('ridge', Ridge(alpha=50)),
        ('rf',    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ]

stacking = StackingRegressor(estimators=base_models, final_estimator=Ridge(alpha=10), cv=5)

# TODO: tính CV RMSE của stacking và thêm vào bảng comparison
# stacking_rmse = rmse_cv(stacking)
# print(f"Stacking CV RMSE = {stacking_rmse:.4f}")

# ── Bài Tập 2: Residuals Ridge vs Lasso ───────────────────
# TODO: vẽ residuals của Ridge và Lasso cạnh nhau
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# ...

### Gợi ý Bài Tập

**Bài 1 — Stacking:**
```python
stacking_rmse = rmse_cv(stacking)
print(f"Stacking CV RMSE = {stacking_rmse:.4f}")
# Thêm vào bảng:
comparison.loc[len(comparison)+1] = ['Stacking', stacking_rmse]
print(comparison.sort_values('CV RMSE'))
```

**Bài 2 — Residuals side by side:**
```python
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, model, name in zip(axes, [ridge, lasso], ['Ridge', 'Lasso']):
    res = y - model.predict(X_train)
    ax.scatter(model.predict(X_train), res, alpha=0.3, s=10)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_title(f'Residuals — {name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Residual')
plt.tight_layout()
plt.show()
```

---

## 🏁 Tổng Kết Buổi 7

| Task | Nội dung | Kết quả |
|------|---------|---------|
| **Metrics** | RMSE, MAE, R² | Biết cách đọc và chọn metric phù hợp |
| **CV phân tích** | RMSE từng fold | Đánh giá độ ổn định của model |
| **Learning Curves** | Train vs Val RMSE | Phát hiện overfit/underfit |
| **Residuals** | Phân phối sai số | Biết model còn thiếu sót ở đâu |
| **So sánh** | Bảng & biểu đồ | Chọn được model tốt nhất |

### Buổi 8 — Kaggle Submission & Cải Thiện

Buổi tới ta sẽ:
- Tạo file submission và submit lên Kaggle
- Phân tích leaderboard score vs CV score
- Các kỹ thuật cải thiện: feature selection nâng cao, blending weights, post-processing